In [ ]:
!pip uninstall -y transformers sentence-transformers peft accelerate
!pip install -q transformers==4.39.3
!pip install -q sentence-transformers==2.7.0
!pip install -q accelerate==0.29.3

In [ ]:
import transformers
import sentence_transformers
import accelerate

print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("Accelerate:", accelerate.__version__)

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import pipeline

print("Imports Successful")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/menu.csv')

print(df.shape)
df.head()

In [ ]:
print(df.columns)

print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
df["combined_text"] = (
    df["dish_name"].astype(str)
    + " | "
    + df["category"].astype(str)
    + " | "
    + df["dietary_tags"].astype(str)
    + " | ₹"
    + df["price"].astype(str)
    + " | "
    + df["description"].astype(str)
)

df[["dish_name", "combined_text"]].head()

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded")

In [ ]:
dish_embeddings = embedding_model.encode(
    df["combined_text"].tolist(),
    convert_to_numpy=True
)

print("Embedding Shape:", dish_embeddings.shape)

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def search(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    similarities = cosine_similarity(
        query_embedding,
        dish_embeddings
    )[0]

    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = df.iloc[top_indices].copy()

    results["similarity_score"] = similarities[top_indices]

    return results[
        [
            "dish_name",
            "category",
            "price",
            "dietary_tags",
            "similarity_score"
        ]
    ]

In [ ]:
search("healthy food")

In [ ]:
from transformers import pipeline

qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad"
)

print("QA Model Loaded")

In [ ]:
def ask(question):

    matches = search(question, top_k=3)

    best_match = matches.iloc[0]["dish_name"]

    context = df[
        df["dish_name"] == best_match
    ]["combined_text"].values[0]

    answer = qa_pipeline(
        question=question,
        context=context
    )

    return {
        "Top Matches": matches,
        "Answer": answer["answer"],
        "Confidence": answer["score"]
    }

In [ ]:
result = ask("Is butter chicken spicy?")

print("Answer:", result["Answer"])
print("Confidence:", result["Confidence"])

display(result["Top Matches"])

In [ ]:
def menu_qa(question):

    result = ask(question)

    print("\nQUESTION:")
    print(question)

    print("\nDIRECT ANSWER:")
    print(result["Answer"])

    print("\nCONFIDENCE:")
    print(round(result["Confidence"], 4))

    print("\nTOP MATCHES:")

    display(result["Top Matches"])

In [ ]:
menu_qa("Suggest a light and nutritious meal")

In [ ]:
while True:

    question = input("\nAsk Menu Question: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    menu_qa(question)

In [ ]:
import re

def filtered_search(query, top_k=5):

    temp_df = df.copy()

    if "vegan" in query.lower():
        temp_df = temp_df[
            temp_df["dietary_tags"]
            .str.contains("vegan", case=False, na=False)
        ]

    price_match = re.search(
        r"under\s+(\d+)",
        query.lower()
    )

    if price_match:
        limit = int(price_match.group(1))

        temp_df = temp_df[
            temp_df["price"] <= limit
        ]

    texts = temp_df["combined_text"].tolist()

    temp_embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True
    )

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    scores = cosine_similarity(
        query_embedding,
        temp_embeddings
    )[0]

    idx = np.argsort(scores)[::-1][:top_k]

    results = temp_df.iloc[idx].copy()

    results["similarity_score"] = scores[idx]

    return results

In [ ]:
filtered_search(
    "vegan dishes under 200"
)